In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA


In [ ]:
crime = pd.read_csv("rms_crime_incidents.csv", comment='#')

crime.head()

In [ ]:
crime['incident_occurred_at'].head()

crime["date"] = pd.to_datetime(
    crime["incident_occurred_at"],
    utc=True
)

crime_ts = crime[
    (crime["incident_occurred_at"] >= "2017-01-01") &
    (crime["incident_occurred_at"] <  "2026-01-01")
]

monthly_counts = (
    crime_ts
    .set_index("date")
    .resample("MS")         # Month Start
    .size()
    .rename("crime_count")
    .reset_index()
)

print("Weekly points:", len(weekly_counts))
print("Monthly points:", len(monthly_counts))

In [ ]:
print(crime['date'].min())
monthly_counts.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(monthly_counts['date'], monthly_counts['crime_count'])
plt.xlabel('Month'); plt.ylabel('Crime Count'); plt.show()

weekly_counts = weekly_counts.iloc[1:-1].reset_index(drop=True)

plt.figure(figsize=(8, 4))
plt.plot(weekly_counts['date'], weekly_counts['crime_count'])
plt.xlabel('Week'); plt.ylabel('Crime Count'); plt.show()


In [ ]:
crime_ts["year"] = crime_ts["date"].dt.year

crime_ts.groupby("year").size()

weekly_counts["crime_count"].describe()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(monthly_counts['crime_count'], ax=ax, lags=40)
plt.tight_layout()
plt.show()



In [ ]:
print(crime_ts['offense_category'].unique())

print(crime_ts['arrest_charge'].nunique())

violent_categories = {
    'ASSAULT',
    'AGGRAVATED ASSAULT',
    'ROBBERY',
    'SEXUAL ASSAULT',
    'SEX OFFENSES',
    'HOMICIDE',
    'JUSTIFIABLE HOMICIDE',
    'KIDNAPPING',
    'ARSON',
    'EXTORTION',
    'FAMILY OFFENSE'
}

crime_ts['violent'] = crime_ts['offense_category'].isin(violent_categories)

crime_ts['crime_type'] = pd.Categorical(
    crime_ts['violent'].map({True: 'Violent', False: 'Non-violent'}),
    categories=['Non-violent', 'Violent']
)

crime_ts['crime_type'].value_counts()

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(monthly_counts['date'], monthly_counts['Violent'], label='Violent')
plt.plot(monthly_counts['date'], monthly_counts['Non-violent'], label='Non-violent')

plt.xlabel('Month')
plt.ylabel('Crime Count')
plt.legend()
plt.tight_layout()
plt.show()